```markdown
# Superstore Sales Analysis and Forecasting

This notebook performs a comprehensive analysis of Superstore sales data, covering data cleaning, exploratory data analysis (EDA), statistical correlation, customer segmentation, and sales forecasting using the Prophet library.

## Table of Contents

1.  [Libraries Import and Dataset Load](#step-1-libraries-import-aur-dataset-load-karna)
2.  [Data Cleaning & Preprocessing](#step-2-data-cleaning--preprocessing-real-world-cleaning)
3.  [Exploratory Data Analysis (EDA) - Business Insights](#step-3-exploratory-data-analysis-eda---business-insights)
    *   Category-wise Sales and Profit Analysis
    *   Monthly Sales Trend (Seasonality)
4.  [Advanced Statistical Correlation](#step-4-advanced-statistical-correlation)
5.  [Customer Segmentation Analysis](#step-5-customer-segmentation-analysis)
6.  [Time Series Analysis (Sales Trend)](#step-6-time-series-analysis-sales-trend)
7.  [RFM Analysis (Professional Customer Scoring)](#step-7-rfm-analysis-professional-customer-scoring)
8.  [Sales Forecasting with Prophet](#step-8-sales-forecasting-with-prophet)
    *   Data Preparation
    *   Model Training
    *   Future Prediction (6 Months)
9.  [Visualization](#step-9-visualization-result-dekhna)

## 1. Libraries Import and Dataset Load
This section imports necessary Python libraries like `pandas`, `numpy`, `matplotlib`, `seaborn`, and `plotly.express` for data manipulation, visualization, and sets up aesthetic preferences for plots. It then loads the `superstore.csv` dataset.

## 2. Data Cleaning & Preprocessing
This step involves checking for missing values, converting 'Order Date' and 'Ship Date' columns to datetime objects, creating new features like 'Order Year', 'Order Month', and 'Order Day', and calculating 'Profit Margin'.

## 3. Exploratory Data Analysis (EDA) - Business Insights
This section delves into key business questions:
*   **Category-wise Sales and Profit Analysis**: Visualizes total sales and profit across different product categories to identify top-performing departments.
*   **Monthly Sales Trend (Seasonality)**: Analyzes how sales fluctuate over months across different years to identify seasonal patterns.

## 4. Advanced Statistical Correlation
Examines the correlation between 'Sales', 'Quantity', 'Discount', and 'Profit' using a heatmap to understand their relationships, particularly the impact of discounts on profit.

## 5. Customer Segmentation Analysis
Analyzes sales, profit, and order counts by customer segments (Consumer, Corporate, Home Office) to determine which segment should be the primary target.

## 6. Time Series Analysis (Sales Trend)
This step sets 'Order Date' as the index, aggregates monthly sales, and visualizes the overall sales trend over the full timeline using an interactive line plot.

## 7. RFM Analysis (Professional Customer Scoring)
Applies Recency, Frequency, Monetary (RFM) analysis to segment customers based on their purchasing behavior. It calculates Recency (days since last order), Frequency (total orders), and Monetary (total sales) values, and visualizes them to map customer value.

## 8. Sales Forecasting with Prophet
Utilizes the Prophet library for time series forecasting.
*   **Data Preparation**: Prepares the sales data into a format suitable for Prophet (columns 'ds' for date and 'y' for sales).
*   **Model Training**: Initializes and trains the Prophet model with yearly and weekly seasonality.
*   **Future Prediction (6 Months)**: Generates a future dataframe for the next 180 days and makes sales predictions.

## 9. Visualization
Visualizes the sales forecast for the next 6 months using interactive plots from Prophet, including the main forecast graph and component plots (trend, weekly, yearly patterns) to understand underlying sales drivers.
```

Step 1: Importing Libraries and Loading the Dataset
Here we are calling the tools that will process and visualize our data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
%matplotlib inline

try:
    df = pd.read_csv('Superstore.csv', encoding='latin1')
except:
    df = pd.read_csv('Superstore.csv', encoding='cp1252')

df.head()

Step 2: Data Cleaning & Preprocessing (Real-world Cleaning)
Real datasets always contain errors. We'll need to format dates correctly and remove unnecessary columns.

In [ ]:
print(df.isnull().sum())

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month_name()
df['Order Day'] = df['Order Date'].dt.day_name()

df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100

print("Data cleaning complete. Ready for analysis.")

Step 3: Exploratory Data Analysis (EDA) - Business Insights
Now we'll address the real business questions.

A. Category-wise Sales and Profit Analysis
We need to see which department is generating the most profit.

In [ ]:
category_analysis = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()

fig = px.bar(category_analysis, x='Category', y=['Sales', 'Profit'],
             title='Total Sales vs Profit by Category',
             barmode='group', labels={'value': 'Amount ($)'})
fig.show()

Step 3.2: Monthly Sales Trend (Seasonal)
Are sales increasing each month?

In [ ]:
monthly_sales = df.groupby(['Order Year', 'Order Month'])['Sales'].sum().reset_index()

plt.figure(figsize=(15, 6))
sns.lineplot(data=monthly_sales, x='Order Month', y='Sales', hue='Order Year', marker='o')
plt.title('Monthly Sales Trend Over Years')
plt.xticks(rotation=45)
plt.show()

Step 4: Advanced Statistical Correlation
Does discounting actually reduce profits? We'll use Hetmap for this.

In [ ]:
plt.figure(figsize=(10, 6))
correlation = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Sales Metrics')
plt.show()

Step 5: Customer Segmentation Analysis
Which segment (consumer, corporate, home office) should we target?

In [ ]:
segment_analysis = df.groupby('Segment').agg({'Sales':'sum', 'Profit':'sum', 'Order ID':'count'}).reset_index()
segment_analysis.rename(columns={'Order ID': 'Order Count'}, inplace=True)

fig_pie = px.pie(segment_analysis, values='Profit', names='Segment',
                 title='Profit Distribution by Customer Segment',
                 hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
fig_pie.show()

Step 6: Time Series Analysis (Sales Trend)
In business, it's crucial to know how sales are trending—are we growing or declining?

In [ ]:
df_trend = df.set_index('Order Date')

monthly_trend = df_trend['Sales'].resample('MS').sum().reset_index()

fig_trend = px.line(monthly_trend, x='Order Date', y='Sales',
                    title='Monthly Sales Trend (Full Timeline)',
                    markers=True, line_shape='spline')

fig_trend.update_layout(xaxis_title='Timeline', yaxis_title='Total Sales ($)')
fig_trend.show()

Step 7: RFM Analysis (Professional Customer Scoring)
This is an advanced marketing technique. We judge customers based on three things:

Recency (R): When did you last order?

Frequency (F): How many times did you order?

Monetary (M): How much money did you spend?

In [ ]:
import datetime as dt

latest_date = df['Order Date'].max() + dt.timedelta(days=1)

rfm = df.groupby('Customer Name').agg({
    'Order Date': lambda x: (latest_date - x.max()).days,
    'Order ID': 'count',
    'Sales': 'sum'
}).reset_index()

rfm.columns = ['Customer Name', 'Recency', 'Frequency', 'Monetary']

top_customers = rfm.sort_values('Monetary', ascending=False).head(10)

fig_rfm = px.scatter(rfm, x='Recency', y='Frequency', size='Monetary',
                     color='Monetary', hover_name='Customer Name',
                     title='RFM Analysis: Customer Value Mapping')
fig_rfm.show()

Step 8.1: Data Preparation (Most Important)
We need to prepare the data for Prophet. We will aggregate daily sales.

In [ ]:
from prophet import Prophet

df_prophet = df.groupby('Order Date')['Sales'].sum().reset_index()
df_prophet.columns = ['ds', 'y']

df_prophet.head()

9. Model Training
Now we will train the Prophet model. We will include 'yearly_seasonal' because retail sales follow a pattern every year.


In [ ]:
model = Prophet(yearly_seasonality=True, daily_seasonality=False, weekly_seasonality=True)

model.fit(df_prophet)

print("Model training complete! Starting preparation for the next 6 months...")

Step 8.3: Future Prediction (6 Months)
We need to create a blank dataframe for the next 180 days (6 months) and ask the model for predictions.

In [ ]:
future = model.make_future_dataframe(periods=180)

forecast = model.predict(future)

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

In [ ]:
# Summary of forecast for next 6 months
# This cell computes simple human-readable metrics and a short paragraph.

# Ensure 'forecast' and 'df_prophet' are available in the namespace
start_date_forecast = df_prophet['ds'].max() + pd.Timedelta(days=1)
end_date_forecast = start_date_forecast + pd.Timedelta(days=179)

forecast['ds'] = pd.to_datetime(forecast['ds'])
future_forecast = forecast[(forecast['ds'] >= start_date_forecast) & (forecast['ds'] <= end_date_forecast)].copy()

# Key metrics
total_predicted_sales_6months = future_forecast['yhat'].sum()
avg_daily_sales = future_forecast['yhat'].mean()
peak_day = future_forecast.loc[future_forecast['yhat'].idxmax(), 'ds']
peak_value = future_forecast['yhat'].max()
trend_direction = 'increasing' if future_forecast['yhat'].iloc[-1] > future_forecast['yhat'].iloc[0] else 'decreasing' if future_forecast['yhat'].iloc[-1] < future_forecast['yhat'].iloc[0] else 'stable'

# Month-level aggregation for peak month
future_forecast['month'] = future_forecast['ds'].dt.to_period('M')
month_sales = future_forecast.groupby('month')['yhat'].sum().reset_index()
peak_month = month_sales.loc[month_sales['yhat'].idxmax(), 'month'].strftime('%Y-%m')

# Print plain-English summary
print('Forecast Summary (Next 6 months):')
print(f'- Estimated total sales: ${total_predicted_sales_6months:,.2f}')
print(f'- Estimated average daily sales: ${avg_daily_sales:,.2f}')
print(f'- Peak forecast day: {pd.to_datetime(peak_day).date()} with predicted sales ${peak_value:,.2f}')
print(f'- Peak forecast month: {peak_month}')
print(f'- Overall short-term trend: {trend_direction}')

print('\nShort summary:')
print(f'Over the next six months, total sales are estimated at ${total_predicted_sales_6months:,.0f}. The daily average is about ${avg_daily_sales:,.0f}. The forecast shows the highest daily sales around {pd.to_datetime(peak_day).date()}, and the busiest month is {peak_month}. The short-term trend is {trend_direction}.')

Step 9: Visualization (Seeing the Results)
Now we've created an interactive graph so you can see whether sales will go up or down over the next 6 months.

## Step 10: Forecast Summary and Business Insights (6 Months)

---

## 11. Conclusion and Recommendations

In this project, we conducted a comprehensive analysis of Superstore sales data. Here are some key findings and business recommendations:

### Key Findings:

* Category-wise Performance:** The Technology and Office Supplies categories consistently showed high sales and profits, while the Furniture category experienced low or even negative profit margins despite high sales. Discounts had a negative impact on the Furniture category's profits.
* Seasonality:** The sales data exhibits clear monthly and yearly seasonality, which the Prophet model captured well. Generally, sales peak at the end of the year (October, November, and December).
* **Customer Segmentation:** RFM analysis has helped identify high-value customers who are most valuable to the business. The Consumer segment contributes the most to overall sales and profits.
* **Discount Impact:** Correlation metrics revealed that discounts have a negative correlation with profits. High discounts often lead to lower profit margins.
* **Future Sales Trend:** The Prophet model has forecasted sales for the next 6 months, which can help businesses with future planning.

### Business Recommendations:

1. **Focus on the Furniture Category:** Develop strategies to improve profit margins in the Furniture category. This could include better deals with suppliers, reducing operational costs, or focusing on high-margin furniture items. Use discounts strategically, especially on furniture.
2. Target High-Value Customers: Launch special loyalty programs, personalized offers, and exclusive services for high-value customers identified through RFM analysis to increase their engagement and spending.
3. Seasonal Marketing Campaigns: Intensify marketing efforts during high-sales months (such as the holiday season). In months with lower sales, try to boost sales through special promotions or new product launches.
4. **Discount Policy Review:** Regularly review the impact of discounts on profits. Use limited and targeted discounts that boost sales but don't harm profit margins.

5. **Inventory Management:** Optimize inventory levels using sales forecasts to avoid stock-outs or excessive inventory costs, especially during peak seasons.

This analysis will help businesses make better decisions and strategize for the future.

## Step 11: Conclusion and Recommendations

In this project, we conducted a comprehensive analysis of Superstore sales data. Here are some key findings and business recommendations:

### Key Findings:

* Category-wise Performance:** The Technology and Office Supplies categories consistently showed high sales and profits, while the Furniture category experienced low or even negative profit margins despite high sales. Discounts had a negative impact on the Furniture category's profits.
* Seasonality:** The sales data exhibits clear monthly and yearly seasonality, which the Prophet model captured well. Generally, sales peak at the end of the year (October, November, and December).
* **Customer Segmentation:** RFM analysis has helped identify high-value customers who are most valuable to the business. The Consumer segment contributes the most to overall sales and profits.
* **Discount Impact:** Correlation metrics revealed that discounts have a negative correlation with profits. High discounts often lead to lower profit margins.
* **Future Sales Trend:** The Prophet model has forecasted sales for the next 6 months, which can help businesses with future planning.

### Business Recommendations:

1. **Focus on the Furniture Category:** Develop strategies to improve profit margins in the Furniture category. This could include better deals with suppliers, reducing operational costs, or focusing on high-margin furniture items. Use discounts strategically, especially on furniture.
2. Target High-Value Customers: Launch special loyalty programs, personalized offers, and exclusive services for high-value customers identified through RFM analysis to increase their engagement and spending.
3. Seasonal Marketing Campaigns: Intensify marketing efforts during high-sales months (such as the holiday season). In months with lower sales, try to boost sales through special promotions or new product launches.
4. **Discount Policy Review:** Regularly review the impact of discounts on profits. Use limited and targeted discounts that boost sales but don't harm profit margins.

5. **Inventory Management:** Optimize inventory levels using sales forecasts to avoid stock-outs or excessive inventory costs, especially during peak seasons.

This analysis will help businesses make better decisions and strategize for the future.

## Project Execution and Methodology Analysis

### 1. Project Overview
This capstone project utilizes the **Superstore Sales Dataset** to drive business intelligence through data science. The project is structured into three main phases:
*   **Phase 1: Foundation** (Cleaning and EDA)
*   **Phase 2: Insights** (Statistical Analysis and Customer Segmentation)
*   **Phase 3: Foresight** (Time Series Forecasting and Strategy)

### 2. Technical Implementation (Commands & Logic)
*   **Data Handling:** Used `pandas` for robust preprocessing. We specifically addressed encoding issues (`latin1`/`cp1252`) to ensure data integrity.
*   **Visualization:** Leveraged `Plotly` for interactive business dashboards and `Seaborn/Matplotlib` for statistical heatmaps.
*   **Forecasting Engine:** Employed the `Prophet` library, configured with yearly and weekly seasonality, to predict future sales trends based on historical patterns.

### 3. Markdown Structure Analysis
*   **Sequential Logic:** The notebook follows a `Step 1` through `Step 11` hierarchy, making it accessible for non-technical stakeholders.
*   **Narrative Flow:** Each code block is preceded by a markdown cell explaining the *Business Context* (the 'Why') and followed by interpretations or visual outputs (the 'What').
*   **Executive Summary:** The project concludes with actionable recommendations, translating data metrics into business growth strategies.

In [ ]:
from prophet.plot import plot_plotly, plot_components_plotly

fig1 = plot_plotly(model, forecast)
fig1.update_layout(title='Sales Forecast: Next 6 Months', xaxis_title='Date', yaxis_title='Sales ($)')
fig1.show()

fig2 = plot_components_plotly(model, forecast)
fig2.show()